In [3]:

# configを読み取る
import datetime
import os
import yaml
import json

config_path = "/workspace/configs/base_config.yaml"

# read config
config = yaml.safe_load(open(config_path, "r"))

# get parameters from config
video_dir = config["video_dir"]
output_dir_top = config["output_dir"]
groundingdino_config_path = config["groundingdino"]["config_path"]
groundingdino_checkpoint = config["groundingdino"]["checkpoint"]

# read annotation
annotation_path = config["annotation_path"]
annotation = [json.loads(line) for line in open(annotation_path, "r").readlines()]

videos_paths = [f"{video_dir}/{a['video_path']}" for a in annotation]
instructions = [a["instruction"] for a in annotation]

# 出力フォルダを作成
# time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
time_stamp = "test"
output_dir = f"{output_dir_top}/{time_stamp}"
os.makedirs(output_dir, exist_ok=True)


# 本番では使わない
subclasses = [a["selected_subclass"] for a in annotation]


In [4]:
# 'selected_subclass': 'Dolly in', の array id 探す
target_subclass = "Dolly in"
target_ids = [i for i, a in enumerate(subclasses) if a == target_subclass]
local_target_id = 0
target_instruction = instructions[target_ids[local_target_id]]
target_video_path  = videos_paths[target_ids[local_target_id]]

print(f"num of target_subclass: {len(target_ids)}")
print(f"target_instruction: {target_instruction}")
print(f"target_video_path: {target_video_path}")


num of target_subclass: 3
target_instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
target_video_path: /workspace/data/videos/wyzi9GNZFMU_0_0to121.mp4


In [ ]:
# instruction parser を読み込む
# /workspace/src にパスを通す
import sys
sys.path.append("/workspace/src")
from parse.instruction_parser_v3_rulebase_trial013_singlefile_kai2 import (
    build_parser
)
parser = build_parser()
request = parser.infer(target_instruction)
action = request["action"]
target = request["target"]
print(f"request: {request}")


In [ ]:
# load video
from utils.video_utility import load_video, write_video, show_video
frames, fps, width, height = load_video(target_video_path)

print(f"num of frames: {len(frames)}")
print(f"fps: {fps}, width: {width}, height: {height}")

# mp4を表示
show_video(target_video_path)


In [ ]:
from postprocess.zoom_in import zoom_in
zoom_factor = 1.0
zoom_target = "face" # "face . person ."
out_frames = zoom_in(frames, width, height, zoom_target=zoom_target, zoom_factor=zoom_factor)

In [ ]:
out_path = f"{output_dir}/zoom_in_output.mp4"
write_video(out_path, out_frames, fps, width, height)

In [ ]:

show_video(out_path)

In [ ]:
frames, fps, width, height = load_video(out_path)
print(f"num of frames: {len(frames)}")
print(f"fps: {fps}, width: {width}, height: {height}")